In [1]:
import pandas as pd
pd.set_option('display.max_rows', 500)

def merge_by_chr_bp_and_flip_alleles(bim, phs, chrcol, bpcol):
    # NB - do not consider strand flips, and retain ambiguous SNPs
    # NB! There are duplicates on CHR:BP, but it's ok as long as we don't have duplicates after merging with phs file

    print(f'{len(phs)} SNPs in original model')

    assert 0==bim['SNP'].duplicated().sum()
    if 'GP' in bim: del bim['GP']

    phs = pd.merge(phs[[chrcol, bpcol, 'Effect', 'Ref', 'Beta']].rename(columns={chrcol:'CHR',bpcol:'BP'}), bim, on=['CHR','BP'], how='left')
    phs=phs[~phs['SNP'].isnull()]
    print(f'{len(phs)} SNPs after merging on CHR:BP')
    
    idx = (phs['A1']==phs['Effect']) & (phs['A2']==phs['Ref'])
    idx2 = (phs['A1']==phs['Ref']) & (phs['A2']==phs['Effect']) 
    
    phs.loc[idx2, 'Beta'] = -phs.loc[idx2, 'Beta'] 
    phs.loc[idx2, ['Effect', 'Ref']] = phs.loc[idx2, ['Ref', 'Effect']].values
    print(f'Flip A1<->A2 and invert sign of BETA for {idx2.sum()} SNPs')
    
    phs = phs[idx | idx2]
    assert (phs['Effect'] == phs['A1']).all()
    assert (phs['Ref'] == phs['A2']).all()
    print(f'{len(phs)} SNPs after matching alleles')
    
    assert not phs['SNP'].duplicated().any()  # check no duplicated SNPs
    return phs


In [7]:
df1=pd.read_csv('../../data/interim/NORGENE/norgene_merged.bim', sep='\t', header=None, names=['CHR','SNP','GP','BP','A1','A2'])
print('highest chromosome: ', df1['CHR'].max()) # often there is no chr23 aka chrX 

phs601=pd.read_excel('01_geneticScore.finalModel.snplist.xlsx')
phs601=merge_by_chr_bp_and_flip_alleles(df1, phs601, 'Chromosome', 'Position (GRCh38)')
phs601.to_csv('phs601_norgene.csv', sep='\t', index=False)


highest chromosome:  22
601 SNPs in original model
515 SNPs after merging on CHR:BP
Flip A1<->A2 and invert sign of BETA for 242 SNPs
513 SNPs after matching alleles


In [8]:
df2=pd.read_csv('/cluster/projects/p212/moba_qc_imputation/BA/release6/moba_imputed_all.bim', sep='\t', header=None, names=['CHR','SNP','GP','BP','A1','A2'])
if 'GP' in df2: del df2['GP']
print('highest chromosome: ', df1['CHR'].max()) # often there is no chr23 aka chrX 

phs601=pd.read_excel('01_geneticScore.finalModel.snplist.xlsx')
phs601=merge_by_chr_bp_and_flip_alleles(df2, phs601, 'Chromosome', 'Position (GRCh37)')
phs601.to_csv('phs601_moba.csv', sep='\t', index=False)


highest chromosome:  22
601 SNPs in original model
527 SNPs after merging on CHR:BP
Flip A1<->A2 and invert sign of BETA for 250 SNPs
527 SNPs after matching alleles
